# Patient Monitoring - Kafka Ingestion Pipeline

This notebook demonstrates:

1. Kafka producer
2. Kafka consumer
3. Pydantic schema validation
4. Invalid records routing to Dead Letter Queue (DLQ)

Architecture:

Producer
    |
    v
Kafka Topic
    |
    v
Consumer
    |
    +--> Valid Records
    |
    +--> DLQ (invalid records + rejection reason)

## Imports

In [19]:
import json
import pandas as pd

from pydantic import BaseModel, ValidationError, field_validator
from kafka import KafkaProducer, KafkaConsumer

## Schema Validation

Pydantic is used as the ingestion boundary.
Only records matching the contract are accepted.

## Data Contract

In [20]:
class PatientReading(BaseModel):
    patient_id: str
    ward: str
    heart_rate: int
    oxygen_level: int
    event_time: str

    @field_validator("heart_rate")
    @classmethod
    def validate_heart_rate(cls, value):
        if value < 20 or value > 250:
            raise ValueError("Invalid heart rate range")
        return value

    @field_validator("oxygen_level")
    @classmethod
    def validate_oxygen_level(cls, value):
        if value < 0 or value > 100:
            raise ValueError("Invalid oxygen level range")
        return value

## Kafka Producer Setup

In [21]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

## Generate Test Data

In [ ]:
records = [
    {
        "patient_id": "P001",
        "ward": "ICU",
        "heart_rate": 82,
        "oxygen_level": 98,
        "event_time": "2026-07-22 10:00:00"
    },
    {
        "patient_id": "P002",
        "ward": "ICU",
        "heart_rate": 110,
        "oxygen_level": 92,
        "event_time": "2026-07-22 10:05:00"
    },
    {
        "patient_id": "P003",
        "ward": "Ward-A",
        "heart_rate": 75,
        "oxygen_level": 99,
        "event_time": "2026-07-22 10:10:00"
    },
    {
        "patient_id": "P004",
        "ward": "Ward-A",
        "heart_rate": 88,
        "oxygen_level": 96,
        "event_time": "2026-07-22 10:15:00"
    },
    {
        "patient_id": "P005",
        "ward": "ICU",
        "heart_rate": 130,
        "oxygen_level": 89,
        "event_time": "2026-07-22 10:20:00"
    },
    {
        "patient_id": "P006",
        "ward": "Ward-B",
        "heart_rate": 70,
        "oxygen_level": 98,
        "event_time": "2026-07-22 10:25:00"
    },
    {
        "patient_id": "P007",
        "ward": "Ward-B",
        "heart_rate": 95,
        "oxygen_level": 94,
        "event_time": "2026-07-22 10:30:00"
    },
    {
        "patient_id": "P008",
        "ward": "ICU",
        "heart_rate": 105,
        "oxygen_level": 91,
        "event_time": "2026-07-22 10:35:00"
    },
    {
        "patient_id": "P009",
        "ward": "Ward-A",
        "heart_rate": 65,
        "oxygen_level": 100,
        "event_time": "2026-07-22 10:40:00"
    },
    {
        "patient_id": "P010",
        "ward": "Ward-B",
        "heart_rate": "HIGH",
        "oxygen_level": 97,
        "event_time": "2026-07-22 10:45:00"
    }
]


for record in records:
    producer.send(
        "patient_readings",
        record
    )

producer.flush()

print("Messages sent")


Messages sent
Messages sent


## consumer Setup

In [23]:
consumer = KafkaConsumer(
    "patient_readings",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    consumer_timeout_ms=5000
)

C:\Users\reems\AppData\Local\Temp\ipykernel_17512\3076173257.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


## DLQ Producer

In [24]:
dlq_producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

## Validation Boundary

In [25]:
valid_records = []


for msg in consumer:

    record = msg.value

    try:

        validated = PatientReading(**record)

        valid_records.append(
            validated.model_dump()
        )

        print("VALID:", record)


    except ValidationError as e:

        dlq_message = {
            "record": record,
            "reason": str(e)
        }

        dlq_producer.send(
            "patient_readings_dlq",
            dlq_message
        )

        print("INVALID:", record)


dlq_producer.flush()

INVALID: {'patient_id': 'P001', 'heart_rate': 82, 'oxygen_level': 98}
INVALID: {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}
VALID: {'patient_id': 'P001', 'ward': 'ICU', 'heart_rate': 82, 'oxygen_level': 98, 'event_time': '2026-07-22 10:00:00'}
INVALID: {'patient_id': 'P002', 'ward': 'ICU', 'heart_rate': 'HIGH', 'oxygen_level': 97, 'event_time': '2026-07-22 10:01:00'}
VALID: {'patient_id': 'P001', 'ward': 'ICU', 'heart_rate': 82, 'oxygen_level': 98, 'event_time': '2026-07-22 10:00:00'}
VALID: {'patient_id': 'P002', 'ward': 'ICU', 'heart_rate': 110, 'oxygen_level': 92, 'event_time': '2026-07-22 10:05:00'}
VALID: {'patient_id': 'P003', 'ward': 'Ward-A', 'heart_rate': 75, 'oxygen_level': 99, 'event_time': '2026-07-22 10:10:00'}
VALID: {'patient_id': 'P004', 'ward': 'Ward-A', 'heart_rate': 88, 'oxygen_level': 96, 'event_time': '2026-07-22 10:15:00'}
VALID: {'patient_id': 'P005', 'ward': 'ICU', 'heart_rate': 130, 'oxygen_level': 89, 'event_time': '2026-07-22 10:20:00'}
VA

In [26]:
valid_df = pd.DataFrame(valid_records)

valid_df.to_csv(
    "../data/validated_patient_readings.csv",
    index=False
)

print("Validated records saved")

Validated records saved


## DLQ Evidence

In [27]:
dlq_consumer = KafkaConsumer(
    "patient_readings_dlq",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    consumer_timeout_ms=5000
)


for msg in dlq_consumer:
    print(msg.value)

C:\Users\reems\AppData\Local\Temp\ipykernel_17512\233827345.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  dlq_consumer = KafkaConsumer(


{'record': {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}, 'reason': "1 validation error for PatientReading\nheart_rate\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='HIGH', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing"}
{'record': {'patient_id': 'P001', 'heart_rate': 82, 'oxygen_level': 98}, 'reason': "2 validation errors for PatientReading\nward\n  Field required [type=missing, input_value={'patient_id': 'P001', 'h... 82, 'oxygen_level': 98}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nevent_time\n  Field required [type=missing, input_value={'patient_id': 'P001', 'h... 82, 'oxygen_level': 98}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing"}
{'record': {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}, 'reason': "3 validation errors for Patient